# Byte Pair Encoding (BPE) for Tokenization

This notebook explains **Byte Pair Encoding (BPE)** from first principles and implements a small, educational tokenizer using only Python's standard library.

## Learning objectives

By the end of the notebook, you should be able to:

1. Explain why language models use tokens instead of treating every word as an indivisible unit.
2. Describe how BPE learns frequent symbol pairs from a training corpus.
3. Train a small BPE vocabulary step by step.
4. Encode new text using the learned merge rules.
5. Explain important differences between this teaching implementation and production tokenizers.

> **Main intuition:** BPE begins with very small units and repeatedly joins the pair of adjacent units that appears most often. Frequent patterns become single tokens, while uncommon words remain combinations of smaller tokens.

## 1. Why tokenize text?

A language model works with numbers, not raw strings. A **tokenizer** divides text into units called *tokens* and maps every token to an integer ID.

There are several possible strategies:

- **Word tokenization:** easy to understand, but the vocabulary becomes enormous and unseen words are a problem.
- **Character tokenization:** can represent almost any text with a small vocabulary, but creates long sequences and loses useful recurring patterns.
- **Subword tokenization:** a compromise. Common words or fragments get their own tokens, while rare words are constructed from smaller pieces.

BPE is a subword-tokenization algorithm. For example, after training, the word `lowest` might be represented as `low` + `est`, while a rarer word may be split into more pieces.

### The BPE learning loop

1. Split every training word into initial symbols (characters in this notebook).
2. Count every pair of adjacent symbols.
3. Find the most frequent pair.
4. Merge that pair everywhere in the corpus.
5. Save the merge rule and repeat.

The **order** of learned rules matters during encoding.

In [ ]:
from collections import Counter
from typing import Dict, List, Tuple

# A symbol pair is represented as a tuple such as ("l", "o").
Pair = Tuple[str, str]

# Each dictionary key is a word represented as a tuple of symbols.
# The value is the number of times that word occurs in the training corpus.
WordVocabulary = Dict[Tuple[str, ...], int]


def build_word_vocabulary(text: str) -> WordVocabulary:
    """Convert text into a frequency table of character-level words.

    The special symbol </w> marks the end of a word. It lets BPE distinguish,
    for example, a sequence at the end of a word from the same sequence in
    the middle of a word. This convention is useful for teaching classic BPE;
    many modern tokenizers represent spaces or word boundaries differently.
    """
    word_counts = Counter(text.lower().split())

    return {
        tuple(list(word) + ["</w>"]): frequency
        for word, frequency in word_counts.items()
    }


# Repeated words are intentional: their frequencies influence which pair wins.
training_text = "low low low low low lowest lowest newer newer newer wider"
word_vocabulary = build_word_vocabulary(training_text)

print("Initial corpus representation:\n")
for symbols, frequency in word_vocabulary.items():
    print(f"{frequency:>2} × {' '.join(symbols)}")

## 2. Count adjacent pairs

Suppose a word currently has the symbols:

`l  o  w  </w>`

Its adjacent pairs are `(l, o)`, `(o, w)`, and `(w, </w>)`. If the word occurs five times, each pair contributes five—not one—to the corpus-wide counts.

Run the next cell and inspect the most frequent pairs. Because ties are possible, this notebook uses a deterministic alphabetical tie-breaker so repeated runs produce the same result.

In [ ]:
def count_adjacent_pairs(vocabulary: WordVocabulary) -> Counter:
    """Count adjacent symbol pairs, weighted by each word's frequency."""
    pair_counts = Counter()

    for symbols, word_frequency in vocabulary.items():
        # zip(symbols, symbols[1:]) aligns each symbol with its right neighbor.
        # Example: ("l", "o", "w") -> ("l", "o"), ("o", "w")
        for left_symbol, right_symbol in zip(symbols, symbols[1:]):
            pair_counts[(left_symbol, right_symbol)] += word_frequency

    return pair_counts


def select_best_pair(pair_counts: Counter) -> Pair:
    """Select the most frequent pair with a deterministic tie-breaker."""
    if not pair_counts:
        raise ValueError("No adjacent pairs remain to merge.")

    # First maximize frequency. For equal frequencies, choose the
    # alphabetically smallest pair so the lesson is reproducible.
    highest_frequency = max(pair_counts.values())
    tied_pairs = [
        pair for pair, frequency in pair_counts.items()
        if frequency == highest_frequency
    ]
    return min(tied_pairs)


pair_counts = count_adjacent_pairs(word_vocabulary)

print("Ten most frequent pairs:")
for pair, frequency in pair_counts.most_common(10):
    print(f"{pair!s:<16} -> {frequency}")

print("\nSelected pair:", select_best_pair(pair_counts))

## 3. Merge one pair

If the chosen pair is `("l", "o")`, BPE replaces adjacent occurrences with the new symbol `"lo"`.

The merge must be applied **left to right** and only to exact adjacent matches. We construct a new symbol list instead of changing the old one while iterating; modifying a list in place would make it easy to skip symbols or merge the wrong positions.

In [ ]:
def merge_pair_in_symbols(symbols: Tuple[str, ...], pair: Pair) -> Tuple[str, ...]:
    """Merge every non-overlapping occurrence of one pair in one word."""
    merged_symbols: List[str] = []
    position = 0

    while position < len(symbols):
        pair_starts_here = (
            position < len(symbols) - 1
            and symbols[position] == pair[0]
            and symbols[position + 1] == pair[1]
        )

        if pair_starts_here:
            # Joining strings creates a new, larger BPE symbol.
            merged_symbols.append(pair[0] + pair[1])
            position += 2  # Both input symbols have been consumed.
        else:
            merged_symbols.append(symbols[position])
            position += 1

    return tuple(merged_symbols)


def merge_pair_in_vocabulary(
    vocabulary: WordVocabulary,
    pair: Pair,
) -> WordVocabulary:
    """Apply one merge rule to all words in the corpus."""
    updated_vocabulary: WordVocabulary = {}

    for symbols, frequency in vocabulary.items():
        merged_word = merge_pair_in_symbols(symbols, pair)
        # += is robust even if two entries ever produce the same symbol tuple.
        updated_vocabulary[merged_word] = (
            updated_vocabulary.get(merged_word, 0) + frequency
        )

    return updated_vocabulary


example = ("l", "o", "w", "</w>")
example_pair = ("l", "o")
print("Before:", example)
print("After: ", merge_pair_in_symbols(example, example_pair))

## 4. Train the BPE model

Training repeats counting and merging. The learned model is simply an **ordered list of merge rules**.

The number of merges is a teaching-friendly stand-in for choosing a target vocabulary size:

- Fewer merges → smaller vocabulary and longer token sequences.
- More merges → larger vocabulary and shorter token sequences on familiar text.

Production systems usually choose a vocabulary size and train on a much larger, carefully prepared corpus.

In [ ]:
def train_bpe(text: str, number_of_merges: int):
    """Learn an ordered sequence of BPE merge rules from text."""
    vocabulary = build_word_vocabulary(text)
    learned_merges: List[Pair] = []

    print("Learning merge rules:\n")

    for step in range(1, number_of_merges + 1):
        pair_counts = count_adjacent_pairs(vocabulary)

        # This can happen when every word has already become one symbol.
        if not pair_counts:
            print("No pairs remain; training stops early.")
            break

        best_pair = select_best_pair(pair_counts)
        best_pair_frequency = pair_counts[best_pair]

        learned_merges.append(best_pair)
        vocabulary = merge_pair_in_vocabulary(vocabulary, best_pair)

        new_symbol = "".join(best_pair)
        print(
            f"Step {step:>2}: merge {best_pair} "
            f"(frequency {best_pair_frequency}) -> {new_symbol!r}"
        )

    return learned_merges, vocabulary


learned_merges, trained_vocabulary = train_bpe(
    training_text,
    number_of_merges=10,
)

print("\nCorpus after training:\n")
for symbols, frequency in trained_vocabulary.items():
    print(f"{frequency:>2} × {' | '.join(symbols)}")

## 5. Encode new text

Training and encoding are different:

- **Training** discovers the merge rules from corpus frequencies.
- **Encoding** starts from the initial symbols and applies the already learned rules in their original order.

We do not search for the most frequent pair while encoding a new sentence. Its frequencies cannot change the trained tokenizer.

A word that never appeared in training can still be represented—as long as its initial characters are supported—because BPE can leave it split into small units. This is a major advantage over a fixed word-level vocabulary.

In [ ]:
def encode_word(word: str, merge_rules: List[Pair]) -> List[str]:
    """Encode one word by applying learned rules in training order."""
    symbols = tuple(list(word.lower()) + ["</w>"])

    for pair in merge_rules:
        symbols = merge_pair_in_symbols(symbols, pair)

    # For readable output, remove the standalone end marker or detach it from
    # a token that absorbed it. A real tokenizer would preserve boundary
    # information according to its own vocabulary convention.
    readable_tokens: List[str] = []
    for symbol in symbols:
        if symbol == "</w>":
            continue
        if symbol.endswith("</w>"):
            readable_tokens.append(symbol.removesuffix("</w>"))
        else:
            readable_tokens.append(symbol)

    return readable_tokens


def encode_text(text: str, merge_rules: List[Pair]) -> List[str]:
    """Encode whitespace-separated text into one flat token sequence."""
    tokens: List[str] = []

    for word in text.split():
        tokens.extend(encode_word(word, merge_rules))

    return tokens


examples = ["low", "lowest", "newer", "widest", "lower"]

for word in examples:
    print(f"{word:<8} -> {encode_word(word, learned_merges)}")

sentence = "lowest newer lower"
print(f"\nSentence: {sentence!r}")
print("Tokens:  ", encode_text(sentence, learned_merges))

## 6. Build token IDs and decode

A model receives integer IDs rather than token strings. After training, a tokenizer builds a stable mapping from every vocabulary token to an integer.

Decoding reverses that mapping and joins token strings. This simplified notebook removed word-boundary markers for readability, so we retain each encoded word separately when demonstrating exact reconstruction.

In [ ]:
# Collect tokens observed in our examples and sort them so IDs are reproducible.
example_token_sequences = [encode_word(word, learned_merges) for word in examples]
all_example_tokens = sorted({
    token
    for token_sequence in example_token_sequences
    for token in token_sequence
})

# Reserve 0 for an unknown token. Production tokenizers usually have several
# special tokens, such as padding, beginning-of-sequence, or end-of-sequence.
token_to_id = {"[UNK]": 0}
token_to_id.update({token: index + 1 for index, token in enumerate(all_example_tokens)})
id_to_token = {token_id: token for token, token_id in token_to_id.items()}


def tokens_to_ids(tokens: List[str]) -> List[int]:
    """Map token strings to IDs, using [UNK] for missing vocabulary items."""
    unknown_id = token_to_id["[UNK]"]
    return [token_to_id.get(token, unknown_id) for token in tokens]


word = "lowest"
tokens = encode_word(word, learned_merges)
token_ids = tokens_to_ids(tokens)
decoded_word = "".join(id_to_token[token_id] for token_id in token_ids)

print("Token-to-ID vocabulary:", token_to_id)
print("Original word:          ", word)
print("BPE tokens:             ", tokens)
print("Integer IDs:            ", token_ids)
print("Decoded word:           ", decoded_word)

assert decoded_word == word, "Encoding followed by decoding should be lossless here."

## 7. Experiments for students

Try changing one variable at a time and explain the result before running the cell again.

1. **Corpus frequency:** add many copies of `wider`. Which rules move earlier?
2. **Merge budget:** compare 0, 5, 10, and 20 merges. How does the token count change?
3. **Unseen words:** encode `newest`, `slowest`, or a word containing a character absent from the corpus.
4. **Remove `</w>`:** what patterns can now merge without knowing where a word ends?
5. **Tie-breaking:** change `select_best_pair`. Can two valid BPE trainings produce different vocabularies?

### Optional exercise

Complete the function below so it returns the intermediate symbols after every merge rule. It is useful for visualizing how a word is gradually compressed.

```python
def trace_encoding(word, merge_rules):
    history = [tuple(list(word.lower()) + ["</w>"])]
    # Apply one rule at a time and append each new symbol tuple to history.
    ...
    return history
```

## Important production details

This implementation is intentionally small. Real tokenizers may also:

- normalize Unicode and whitespace;
- pre-tokenize text with rules or regular expressions;
- start from **bytes** rather than Unicode characters (byte-level BPE);
- include special tokens and explicit unknown-token behavior;
- train on billions of characters with optimized data structures;
- preserve spaces or word boundaries using symbols such as `Ġ` or `▁`;
- package normalization, vocabulary, merge ranks, and decoding rules together.

### Takeaway

BPE learns a vocabulary by repeatedly replacing frequent adjacent symbol pairs. It keeps common text compact without requiring every possible word to be stored as a whole token.